# 🔎 Module 17 — Regular Expressions & Text Processing

## 🎯 What you will learn
1. What regex is; the `re` module functions
2. Pattern syntax: literals, character classes, quantifiers, anchors, groups, alternation
3. `match` vs `search` vs `fullmatch` vs `findall` vs `finditer` vs `sub` vs `split`
4. Groups: capturing, named, non-capturing, backreferences
5. Greedy vs lazy matching; lookahead / lookbehind
6. Flags; compiled patterns
7. Real-world recipes: email, phone, dates, passwords, log parsing, cleaning text
8. Hidden facts & tricks

**Exam relevance:** ⭐⭐⭐ (short `re` questions; very useful in real life)

---
## 1. What is a regular expression?
A **pattern language** for describing text. Use raw strings `r"..."` so backslashes reach `re` untouched.

In [1]:
import re
text = "Contact: asha@mail.com, ravi_k@site.org  Phone: 98765-43210"

print(re.search(r"\d+", text))                 # first match object
print(re.search(r"\d+", text).group())
print(re.findall(r"\w+@\w+\.\w+", text))       # all emails (simple)
print(re.sub(r"\d", "X", text))                # replace digits
print(re.split(r"[,:]\s*", text))              # split on commas/colons

<re.Match object; span=(48, 53), match='98765'>
98765
['asha@mail.com', 'ravi_k@site.org']
Contact: asha@mail.com, ravi_k@site.org  Phone: XXXXX-XXXXX
['Contact', 'asha@mail.com', 'ravi_k@site.org  Phone', '98765-43210']


## 2. The pattern language

### Characters and classes
| Pattern | Matches |
|---|---|
| `abc` | literal "abc" |
| `.` | any character except newline |
| `\d` `\D` | digit / non-digit |
| `\w` `\W` | word char `[A-Za-z0-9_]` / non-word |
| `\s` `\S` | whitespace / non-whitespace |
| `[abc]` | one of a, b, c |
| `[^abc]` | anything except a, b, c |
| `[a-z0-9]` | ranges |
| `\.` `\\` `\(` | escaped literal special characters |

### Quantifiers
| | |
|---|---|
| `*` | 0 or more |
| `+` | 1 or more |
| `?` | 0 or 1 |
| `{n}` | exactly n |
| `{n,m}` | n to m |
| `{n,}` | n or more |

### Anchors and boundaries
`^` start of string, `$` end of string, `\b` word boundary, `\B` non-boundary, `\A` `\Z` absolute start/end

### Groups and alternation
`(abc)` capture, `(?:abc)` non-capturing, `(?P<name>...)` named, `a|b` either, `\1` backreference

In [2]:
tests = [
    (r"colou?r", ["color", "colour", "colr"]),          # ? optional
    (r"^\d{3}-\d{4}$", ["555-1234", "5551234", "555-12345"]),
    (r"^[A-Z][a-z]+$", ["Python", "python", "PYTHON"]),
    (r"\bcat\b", ["cat", "concat", "the cat sat"]),      # whole word only
    (r"gr[ae]y", ["gray", "grey", "groy"]),
    (r"^(yes|no)$", ["yes", "no", "maybe"]),
]
for pat, samples in tests:
    print(f"{pat:22}", [(s, bool(re.search(pat, s))) for s in samples])

colou?r                [('color', True), ('colour', True), ('colr', False)]
^\d{3}-\d{4}$          [('555-1234', True), ('5551234', False), ('555-12345', False)]
^[A-Z][a-z]+$          [('Python', True), ('python', False), ('PYTHON', False)]
\bcat\b                [('cat', True), ('concat', False), ('the cat sat', True)]
gr[ae]y                [('gray', True), ('grey', True), ('groy', False)]
^(yes|no)$             [('yes', True), ('no', True), ('maybe', False)]


## 3. The main functions

| Function | Purpose | Returns |
|---|---|---|
| `re.match(p, s)` | match at the **start** of s | Match or None |
| `re.search(p, s)` | first match **anywhere** | Match or None |
| `re.fullmatch(p, s)` | pattern must match the **whole** string | Match or None |
| `re.findall(p, s)` | all matches | list of str (or tuples if groups) |
| `re.finditer(p, s)` | all matches lazily | iterator of Match objects |
| `re.sub(p, repl, s, count=0)` | replace | new str |
| `re.split(p, s)` | split on pattern | list |
| `re.compile(p)` | precompile for reuse | Pattern |
| `re.escape(s)` | escape special chars in a literal | str |

In [3]:
s = "Order 66 shipped 2025-01-15, order 7 pending"
print(re.match(r"Order", s) is not None, re.match(r"66", s))     # match anchors at start
print(re.search(r"\d+", s).span())
print(re.fullmatch(r"\d+", "12345"), re.fullmatch(r"\d+", "123a"))
print(re.findall(r"\d+", s))
for m in re.finditer(r"[Oo]rder (\d+)", s):
    print(m.group(), "| id:", m.group(1), "| at", m.start(), "-", m.end())

m = re.search(r"(\d{4})-(\d{2})-(\d{2})", s)
print(m.groups(), m.group(0), m.group(1), m.groups(default=""))
print(re.sub(r"(\d{4})-(\d{2})-(\d{2})", r"\3/\2/\1", s))       # reorder date using \1 \2 \3
print(re.sub(r"\d+", lambda m: str(int(m.group()) * 2), s))     # function as replacement!
print(re.split(r"\s*[,;]\s*", "a , b;c ,d"))
print(re.split(r"(,)", "a,b"))                                    # capture group keeps the separators
print(re.escape("a.b*c"))

True None
(6, 8)
<re.Match object; span=(0, 5), match='12345'> None
['66', '2025', '01', '15', '7']
Order 66 | id: 66 | at 0 - 8
order 7 | id: 7 | at 29 - 36
('2025', '01', '15') 2025-01-15 2025 ('2025', '01', '15')
Order 66 shipped 15/01/2025, order 7 pending
Order 132 shipped 4050-2-30, order 14 pending
['a', 'b', 'c', 'd']
['a', ',', 'b']
a\.b\*c


## 4. Groups in depth

In [4]:
# Named groups
m = re.search(r"(?P<user>[\w.]+)@(?P<domain>[\w.]+)\.(?P<tld>[a-z]{2,})", "mail me: kaushik.s@college.edu")
print(m.group("user"), m.group("domain"), m.group("tld"), m.groupdict())

# findall with groups returns tuples
print(re.findall(r"(\w+)=(\d+)", "a=1, b=22, c=333"))
print(dict(re.findall(r"(\w+)=(\d+)", "a=1, b=22")))

# Non-capturing group (?:...)
print(re.findall(r"(?:Mr|Ms|Dr)\. (\w+)", "Mr. Smith met Dr. Jones and Ms. Lee"))

# Backreference: repeated words / repeated letters
print(re.findall(r"\b(\w+) \1\b", "this is is a test test of the the system"))
print(bool(re.search(r"(\w)\1", "hello")), bool(re.search(r"(\w)\1", "world")))

kaushik.s college edu {'user': 'kaushik.s', 'domain': 'college', 'tld': 'edu'}
[('a', '1'), ('b', '22'), ('c', '333')]
{'a': '1', 'b': '22'}
['Smith', 'Jones', 'Lee']
['is', 'test', 'the']
True False


## 5. Greedy vs lazy

Quantifiers are **greedy** by default (match as much as possible). Add `?` to make them **lazy** (as little as possible).

In [5]:
html = "<b>bold</b> and <i>italic</i>"
print(re.findall(r"<.+>", html))          # greedy: swallows everything
print(re.findall(r"<.+?>", html))         # lazy: individual tags
print(re.findall(r"<(\w+)>(.*?)</\1>", html))   # tag name + content, with a backreference

['<b>bold</b> and <i>italic</i>']
['<b>', '</b>', '<i>', '</i>']
[('b', 'bold'), ('i', 'italic')]


## 6. Lookahead & lookbehind (zero-width assertions)

| Syntax | Meaning |
|---|---|
| `X(?=Y)` | X followed by Y |
| `X(?!Y)` | X **not** followed by Y |
| `(?<=Y)X` | X preceded by Y |
| `(?<!Y)X` | X **not** preceded by Y |

In [6]:
prices = "cost: $45, fee: €12, tip: $3"
print(re.findall(r"(?<=\$)\d+", prices))            # digits after a $
print(re.findall(r"\d+(?=\s*dollars)", "5 dollars, 8 euros, 12 dollars"))
print(re.findall(r"\b\w+(?!ing)\b", "run running jump jumping")[:2])

# Insert thousands separators
print(re.sub(r"\B(?=(\d{3})+(?!\d))", ",", "1234567890"))

# Password rule: >=8 chars, a digit, an upper, a lower, a symbol
strong = re.compile(r"^(?=.*\d)(?=.*[a-z])(?=.*[A-Z])(?=.*[^\w\s]).{8,}$")
for pw in ("password", "Passw0rd", "P@ssw0rd!", "sh0rt!A"):
    print(f"{pw:12}", bool(strong.match(pw)))

['45', '3']
['5', '12']
['run', 'running']
1,234,567,890
password     False
Passw0rd     False
P@ssw0rd!    True
sh0rt!A      False


## 7. Flags

| Flag | Short | Effect |
|---|---|---|
| `re.IGNORECASE` | `re.I` | case-insensitive |
| `re.MULTILINE` | `re.M` | `^`/`$` match at each line |
| `re.DOTALL` | `re.S` | `.` matches newline too |
| `re.VERBOSE` | `re.X` | allow whitespace & comments in the pattern |
| `re.ASCII` | `re.A` | `\w` etc. ASCII only |

In [7]:
print(re.findall(r"^\w+", "one two\nthree four", re.M))
print(re.findall(r"python", "Python PYTHON python", re.I))
print(re.search(r"a.b", "a\nb"), re.search(r"a.b", "a\nb", re.S) is not None)

phone = re.compile(r"""
    ^\(?(\d{3})\)?      # area code, optional parentheses
    [-.\s]?             # separator
    (\d{3})             # exchange
    [-.\s]?
    (\d{4})$            # line number
""", re.VERBOSE)
for p in ("(555) 123-4567", "555.123.4567", "5551234567", "55-1234"):
    m = phone.match(p)
    print(f"{p:16}", m.groups() if m else None)

['one', 'three']
['Python', 'PYTHON', 'python']
None True
(555) 123-4567   ('555', '123', '4567')
555.123.4567     ('555', '123', '4567')
5551234567       ('555', '123', '4567')
55-1234          None


## 8. Real-world recipes

In [8]:
# 1. Validate an email (practical version, not the full RFC monster)
email = re.compile(r"^[\w.+-]+@[A-Za-z0-9-]+(\.[A-Za-z0-9-]+)*\.[A-Za-z]{2,}$")
for e in ("a.b@site.com", "bad@", "x@y.co.in", "no at.com"):
    print(f"{e:14}", bool(email.match(e)))

# 2. Extract all dates in several formats
log = "Deploy on 2025-03-01; hotfix 15/03/2025; review 3-Apr-2025"
print(re.findall(r"\d{4}-\d{2}-\d{2}|\d{2}/\d{2}/\d{4}|\d{1,2}-[A-Z][a-z]{2}-\d{4}", log))

# 3. Parse a log file line
line = '192.168.1.10 - - [12/Mar/2025:10:15:32 +0000] "GET /index.html HTTP/1.1" 200 5123'
pat = re.compile(r'(?P<ip>[\d.]+) .*?\[(?P<ts>[^\]]+)\] "(?P<method>\w+) (?P<path>\S+) [^"]*" (?P<status>\d{3}) (?P<size>\d+)')
print(pat.match(line).groupdict())

# 4. Clean text: collapse whitespace, strip punctuation, extract hashtags/mentions/URLs
tweet = "Loving   #Python!!  Thanks @guido  https://python.org/downloads?x=1 #coding"
print(re.sub(r"\s+", " ", tweet))
print(re.findall(r"#\w+", tweet), re.findall(r"@\w+", tweet), re.findall(r"https?://\S+", tweet))

# 5. Tokenise a simple arithmetic expression
tokens = re.findall(r"\d+\.?\d*|[-+*/()]", "3.5*(2+10)-4")
print(tokens)

# 6. camelCase <-> snake_case
def snake(s): return re.sub(r"(?<!^)(?=[A-Z])", "_", s).lower()
def camel(s): return re.sub(r"_(\w)", lambda m: m.group(1).upper(), s)
print(snake("myVariableName"), camel("my_variable_name"))

# 7. Word counting ignoring punctuation
words = re.findall(r"[A-Za-z']+", "It's a dog-eat-dog world, isn't it?")
print(words)

# 8. Mask credit-card / phone numbers
print(re.sub(r"\d(?=\d{4})", "*", "4111111111111111"))

# 9. Template substitution
tpl = "Hello {name}, you owe {amt}."
vals = {"name": "Asha", "amt": "$5"}
print(re.sub(r"\{(\w+)\}", lambda m: vals[m.group(1)], tpl))

# 10. Split a sentence on ., ?, ! keeping the delimiter
print(re.split(r"(?<=[.!?])\s+", "Hi there. How are you? Great! Bye."))

a.b@site.com   True
bad@           False
x@y.co.in      True
no at.com      False
['2025-03-01', '15/03/2025', '3-Apr-2025']
{'ip': '192.168.1.10', 'ts': '12/Mar/2025:10:15:32 +0000', 'method': 'GET', 'path': '/index.html', 'status': '200', 'size': '5123'}
Loving #Python!! Thanks @guido https://python.org/downloads?x=1 #coding
['#Python', '#coding'] ['@guido'] ['https://python.org/downloads?x=1']
['3.5', '*', '(', '2', '+', '10', ')', '-', '4']
my_variable_name myVariableName
["It's", 'a', 'dog', 'eat', 'dog', 'world', "isn't", 'it']
************1111
Hello Asha, you owe $5.
['Hi there.', 'How are you?', 'Great!', 'Bye.']


## 9. Compiling patterns for speed and clarity

In [9]:
pattern = re.compile(r"(\w+)@(\w+)\.com")     # compile once, reuse many times
print(pattern.findall("a@x.com b@y.com"))
print(pattern.pattern, pattern.flags != 0, pattern.groups)
print(pattern.sub(r"\2:\1", "a@x.com"))

[('a', 'x'), ('b', 'y')]
(\w+)@(\w+)\.com True 2
x:a


---
# 🕵️ Hidden Facts & Pro Tricks

In [10]:
# 1. Test patterns interactively: print the match and its span to debug quickly
def show(pat, s):
    for m in re.finditer(pat, s):
        print(f"{m.group()!r} at {m.span()}")
show(r"\b\w{5}\b", "hello there wonderful world")

# 2. re.sub with a dict-driven multi-replace (one pass, no cascading replacements)
repl = {"cat": "dog", "dog": "cat"}
print(re.sub("|".join(map(re.escape, repl)), lambda m: repl[m.group()], "cat chased dog"))

# 3. Catastrophic backtracking: nested quantifiers on failing input can freeze your program
#    e.g. (a+)+$ against "aaaa...b". Keep patterns simple or use possessive/atomic groups (Python 3.11+: (?>...) and a++)
import time
t = time.perf_counter()
re.match(r"(a+)+$", "a" * 22 + "b")
print("nested quantifier took", round(time.perf_counter() - t, 3), "s (grows exponentially with length!)")
t = time.perf_counter()
re.match(r"a++$", "a" * 5000 + "b")           # possessive: no backtracking
print("possessive took", round(time.perf_counter() - t, 5), "s")

# 4. str methods are FASTER and clearer for simple jobs: startswith, in, split, replace
# 5. re.fullmatch replaces the ^...$ boilerplate
print(bool(re.fullmatch(r"\d{3}", "123")))

# 6. Unicode-aware by default: \w matches letters like é 你
print(re.findall(r"\w+", "café 你好 naïve"))

# 7. Use named groups + groupdict() to build dicts/dataclasses straight from text
# 8. `regex` (third-party) adds fuzzy matching, POSIX classes and full Unicode properties
# 9. regex101.com and regexr.com visualise & explain any pattern - use them while learning
# 10. re.DEBUG flag shows how Python parses your pattern
# 11. Escaping user input before putting it into a pattern (avoid regex injection):
user = "1+1=2?"
print(bool(re.search(re.escape(user), "is 1+1=2? yes")))

'hello' at (0, 5)
'there' at (6, 11)
'world' at (22, 27)
dog chased cat
nested quantifier took 0.384 s (grows exponentially with length!)
possessive took 0.00019 s
True
['café', '你好', 'naïve']
True


**Hidden fact:** regular expressions come from theory of formal languages (Stephen Kleene, 1951). "Regular" languages can be recognised by finite automata — yet modern `re` supports backreferences, which makes it *more* powerful (and slower in the worst case) than true regular expressions.

**Hidden fact:** you can't parse HTML/JSON/nested brackets fully with regex — use `html.parser`, `BeautifulSoup`, `json`, or a real parser.

---
# 📝 Exam Corner

**Theory**
1. What is a regular expression? Explain `match()` vs `search()`.
2. Explain the meaning of `\d`, `\w`, `\s`, `.`, `*`, `+`, `?`, `^`, `$`.
3. Write a regex for: a 10-digit mobile number; a valid email; a vehicle number like `KA01AB1234`.
4. What are capturing groups? What does `findall()` return with groups?
5. What is greedy vs non-greedy?

**Output prediction**

In [11]:
print(re.match(r"a", "banana"))
print(re.search(r"a", "banana").start())
print(re.findall(r"a.", "banana"))
print(re.sub(r"a", "A", "banana", count=2))
print(re.split(r"n", "banana"))
print(re.findall(r"(a)(n)", "banana"))
print(bool(re.fullmatch(r"[A-Z]{2}\d{2}[A-Z]{2}\d{4}", "KA01AB1234")))

None
1
['an', 'an']
bAnAna
['ba', 'a', 'a']
[('a', 'n'), ('a', 'n')]
True


<details><summary>💡 Answers</summary>

```
None
1
['an', 'an']
bAnAna
['ba', 'a', 'a']
[('a', 'n'), ('a', 'n')]
True
```
`re.match` fails because "banana" doesn't *start* with "a". `findall(r"a.")` needs an `a` **plus one more character**, so the final lone `a` (nothing after it) is not matched. `split` on `n` gives `['ba', 'a', 'a']`.
</details>

---
## ✅ Summary
- Always use raw strings. `search` anywhere, `match` at the start, `fullmatch` entire string.
- Know classes (`\d \w \s`), quantifiers (`* + ? {n,m}`), anchors, groups, alternation.
- Greedy vs lazy (`.*?`), lookarounds, flags (`re.I re.M re.S re.X`), compile for reuse.
- Prefer simple string methods when they suffice.

➡️ **Next:** Module 18 — NumPy & Pandas for Data Handling.